# NB_02_Silver_Clean
Turns raw bronze rows into typed, de-duplicated, validated tables.

| Step | What happens |
|---|---|
| Incremental read | Only bronze batches newer than the watermark in `ctl_watermark` |
| Standardise | Trim, fix casing, parse two date formats, strip thousands separators |
| De-duplicate | Keep the latest version of each business key |
| Validate | Rows that break a rule go to `silver_dq_rejects` with the reason, never silently dropped |
| Upsert | Delta `MERGE` on the business key, so re-sent rows (e.g. a shipment that has now arrived) update in place |

In [ ]:
table_format = "delta"

In [ ]:
from pyspark.sql import functions as F, Window as W
from delta.tables import DeltaTable

spark.conf.set("spark.sql.ansi.enabled", "false")   # unparseable values become NULL and are caught by the rules below

## Helpers

In [ ]:
def upsert(df, table, keys):
    """Insert new keys, update existing ones."""
    if not spark.catalog.tableExists(table):
        df.write.format(table_format).saveAsTable(table)
        return
    cond = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in keys)
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


def get_watermark(table):
    if not spark.catalog.tableExists("ctl_watermark"):
        return ""
    row = spark.table("ctl_watermark").filter(F.col("table_name") == table).first()
    return row["last_batch"] if row else ""


def set_watermark(table, batch):
    df = (spark.createDataFrame([(table, batch)], "table_name string, last_batch string")
          .withColumn("updated_at", F.current_timestamp()))
    upsert(df, "ctl_watermark", ["table_name"])


def parse_date(c):
    """Sources send ISO dates and UK dd/MM/yyyy dates in the same column."""
    c = F.trim(F.col(c))
    return F.coalesce(F.to_date(c, "yyyy-MM-dd"), F.to_date(c, "dd/MM/yyyy"))


def money(c):
    return F.regexp_replace(F.trim(F.col(c)), ",", "").cast("decimal(18,2)")


def clean_id(c):
    return F.upper(F.trim(F.col(c)))


def latest_per_key(df, keys, order_cols):
    w = W.partitionBy(*keys).orderBy(*order_cols)
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def process(entity, keys, transform, rules=None, order_cols=None):
    """bronze_<entity> -> silver_<entity>, with rejects logged. `rules` is {reason: bad-row condition}."""
    target = f"silver_{entity}"
    wm = get_watermark(target)
    src = spark.table(f"bronze_{entity}").filter(F.col("_batch") > wm)
    max_batch = src.agg(F.max("_batch")).first()[0]
    if max_batch is None:
        print(f"{target:28} no new batches")
        return

    df = transform(src)
    df = latest_per_key(df, keys, order_cols or [F.col("_batch").desc(), F.col("_ingested_at").desc()])

    reason = F.lit(None).cast("string")
    for name, bad in reversed(list((rules or {}).items())):
        reason = F.when(bad, F.lit(name)).otherwise(reason)
    df = df.withColumn("_reject_reason", reason).cache()

    rejects = df.filter(F.col("_reject_reason").isNotNull())
    good = df.filter(F.col("_reject_reason").isNull()).drop("_reject_reason", "_source_file", "_ingested_at")

    n_bad = rejects.count()
    if n_bad:
        (rejects.select(
            F.lit(entity).alias("entity"),
            F.concat_ws("|", *[F.col(k).cast("string") for k in keys]).alias("record_key"),
            F.col("_reject_reason").alias("reason"),
            "_batch",
            F.to_json(F.struct(*[c for c in df.columns if not c.startswith("_")])).alias("payload"),
            F.current_timestamp().alias("rejected_at"))
         .write.format(table_format).mode("append").saveAsTable("silver_dq_rejects"))

    n_good = good.count()
    upsert(good, target, keys)
    set_watermark(target, max_batch)
    df.unpersist()
    print(f"{target:28} upserted {n_good:>7,}   rejected {n_bad:>5,}   watermark {max_batch}")

## Reference data

In [ ]:
process("suppliers", ["supplier_id"], lambda d: d.select(
    clean_id("supplier_id").alias("supplier_id"), F.trim("supplier_name").alias("supplier_name"),
    F.trim("country").alias("country"), clean_id("currency").alias("currency"),
    F.col("payment_terms_days").cast("int").alias("payment_terms_days"),
    "_batch", "_source_file", "_ingested_at"))

process("products", ["sku"], lambda d: d.select(
    clean_id("sku").alias("sku"), F.trim("product_name").alias("product_name"), F.trim("category").alias("category"),
    clean_id("supplier_id").alias("supplier_id"), F.col("unit_cost_ccy").cast("decimal(18,2)").alias("unit_cost_ccy"),
    money("list_price_gbp").alias("list_price_gbp"), F.col("units_per_carton").cast("int").alias("units_per_carton"),
    "_batch", "_source_file", "_ingested_at"))

process("customers", ["customer_id"], lambda d: d.select(
    clean_id("customer_id").alias("customer_id"), F.trim("customer_name").alias("customer_name"),
    F.trim("segment").alias("segment"), F.trim("region").alias("region"),
    F.col("credit_terms_days").cast("int").alias("credit_terms_days"),
    money("credit_limit_gbp").alias("credit_limit_gbp"), parse_date("created_date").alias("created_date"),
    "_batch", "_source_file", "_ingested_at"))

process("fx_rates", ["rate_date", "currency"], lambda d: d.select(
    parse_date("rate_date").alias("rate_date"), clean_id("currency").alias("currency"),
    F.col("rate_to_gbp").cast("double").alias("rate_to_gbp"),
    "_batch", "_source_file", "_ingested_at"),
    rules={"invalid_rate": F.col("rate_to_gbp").isNull() | (F.col("rate_to_gbp") <= 0)})

## Transactions
Referential checks run against the silver reference tables loaded above.

In [ ]:
known_skus = [r[0] for r in spark.table("silver_products").select("sku").collect()]
known_suppliers = [r[0] for r in spark.table("silver_suppliers").select("supplier_id").collect()]
known_customers = [r[0] for r in spark.table("silver_customers").select("customer_id").collect()]

process("purchase_orders", ["po_id"], lambda d: d.select(
    clean_id("po_id").alias("po_id"), parse_date("po_date").alias("po_date"),
    clean_id("supplier_id").alias("supplier_id"), clean_id("sku").alias("sku"),
    F.col("qty_ordered").cast("int").alias("qty_ordered"), F.col("unit_cost").cast("decimal(18,2)").alias("unit_cost"),
    clean_id("currency").alias("currency"), parse_date("expected_ship_date").alias("expected_ship_date"),
    "_batch", "_source_file", "_ingested_at"),
    rules={
        "missing_or_invalid_qty": F.col("qty_ordered").isNull() | (F.col("qty_ordered") <= 0),
        "invalid_date": F.col("po_date").isNull(),
        "unknown_sku": ~F.col("sku").isin(known_skus),
        "unknown_supplier": ~F.col("supplier_id").isin(known_suppliers),
    })

# The forwarder re-sends a shipment when it arrives. Newest batch wins; inside one batch "Arrived" beats "In Transit".
process("shipments", ["shipment_id"], lambda d: d.select(
    clean_id("shipment_id").alias("shipment_id"), clean_id("po_id").alias("po_id"),
    F.trim("container_no").alias("container_no"), parse_date("ship_date").alias("ship_date"),
    parse_date("eta_date").alias("eta_date"), parse_date("arrival_date").alias("arrival_date"),
    F.col("qty_received").cast("int").alias("qty_received"), money("freight_cost_gbp").alias("freight_cost_gbp"),
    money("duty_gbp").alias("duty_gbp"), F.trim("port").alias("port"), F.trim("status").alias("status"),
    "_batch", "_source_file", "_ingested_at"),
    rules={"invalid_date": F.col("ship_date").isNull() | F.col("eta_date").isNull()},
    order_cols=[F.col("_batch").desc(), F.col("arrival_date").desc_nulls_last()])

process("invoice_lines", ["invoice_id", "line_no"], lambda d: d.select(
    clean_id("invoice_id").alias("invoice_id"), F.col("line_no").cast("int").alias("line_no"),
    parse_date("invoice_date").alias("invoice_date"), clean_id("customer_id").alias("customer_id"),
    clean_id("sku").alias("sku"), F.col("qty").cast("int").alias("qty"),
    money("unit_price_gbp").alias("unit_price_gbp"), F.col("discount_pct").cast("int").alias("discount_pct"),
    parse_date("due_date").alias("due_date"),
    "_batch", "_source_file", "_ingested_at"),
    rules={
        "non_positive_qty": F.col("qty").isNull() | (F.col("qty") <= 0),
        "invalid_date": F.col("invoice_date").isNull() | F.col("due_date").isNull(),
        "unknown_sku": ~F.col("sku").isin(known_skus),
        "unknown_customer": ~F.col("customer_id").isin(known_customers),
    })

process("bank_transactions", ["txn_id"], lambda d: d.select(
    clean_id("txn_id").alias("txn_id"), parse_date("txn_date").alias("txn_date"),
    money("amount_gbp").alias("amount_gbp"), F.trim("payer_name").alias("payer_name"),
    F.coalesce(F.trim("reference"), F.lit("")).alias("reference"),
    "_batch", "_source_file", "_ingested_at"),
    rules={
        "invalid_date": F.col("txn_date").isNull(),
        "invalid_amount": F.col("amount_gbp").isNull(),
    })

In [ ]:
spark.table("silver_dq_rejects").groupBy("entity", "reason").count().orderBy("entity", "reason").show(truncate=False)